In [10]:
import os
import cv2
import mediapipe as mp

# ==== CONFIGURATION ====
RAW_DATA_PATH = r"C:\Users\USER\Documents\ML and Apps Project\Drowsiness_dataset_new\Drowsiness_dataset_new\train"
OUTPUT_PATH = r"C:\Users\USER\Documents\ML and Apps Project\eye_dataset"
IMG_SIZE = (64, 64)

# ==== SETUP OUTPUT FOLDERS ====
os.makedirs(os.path.join(OUTPUT_PATH, "open"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "closed"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "unsorted"), exist_ok=True)

# ==== MEDIAPIPE SETUP ====
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True)

eye_indices = {
    'left': [33, 133],
    'right': [362, 263]
}

# ==== EYE CROP FUNCTION ====
def extract_eyes(image):
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    eyes = []

    if not results.multi_face_landmarks:
        return eyes

    h, w, _ = image.shape
    for face_landmarks in results.multi_face_landmarks:
        for side in ['left', 'right']:
            pt1 = face_landmarks.landmark[eye_indices[side][0]]
            pt2 = face_landmarks.landmark[eye_indices[side][1]]

            x1, y1 = int(pt1.x * w), int(pt1.y * h)
            x2, y2 = int(pt2.x * w), int(pt2.y * h)

            margin = 10
            x1, y1 = max(x1 - margin, 0), max(y1 - margin, 0)
            x2, y2 = min(x2 + margin, w), min(y2 + margin, h)

            eye = image[y1:y2, x1:x2]
            if eye.size > 0:
                eyes.append(eye)

    return eyes

# ==== PROCESS A FOLDER ====
def process_folder(folder_name, label_folder=None):
    folder_path = os.path.join(RAW_DATA_PATH, folder_name)
    for i, file_name in enumerate(os.listdir(folder_path)):
        img_path = os.path.join(folder_path, file_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        eyes = extract_eyes(img)
        for j, eye in enumerate(eyes):
            eye_resized = cv2.resize(eye, IMG_SIZE)
            eye_gray = cv2.cvtColor(eye_resized, cv2.COLOR_BGR2GRAY)
            if label_folder:
                save_path = os.path.join(OUTPUT_PATH, label_folder, f"{folder_name}_{i}_{j}.jpg")
            else:
                save_path = os.path.join(OUTPUT_PATH, "unsorted", f"{folder_name}_{i}_{j}.jpg")
            cv2.imwrite(save_path, eye_gray)

# ==== RUN PROCESSING ====

# 1. Crop and label from no_yawn → open
process_folder("no_yawn", label_folder="open")

# 2. Crop and label from yawn → unsorted
process_folder("yawn", label_folder=None)

# 3. Copy existing Open → open
open_src = os.path.join(RAW_DATA_PATH, "Open")
for fname in os.listdir(open_src):
    src_path = os.path.join(open_src, fname)
    dst_path = os.path.join(OUTPUT_PATH, "open", fname)
    img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        cv2.imwrite(dst_path, img)

# 4. Copy existing Closed → closed
closed_src = os.path.join(RAW_DATA_PATH, "Closed")
for fname in os.listdir(closed_src):
    src_path = os.path.join(closed_src, fname)
    dst_path = os.path.join(OUTPUT_PATH, "closed", fname)
    img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        cv2.imwrite(dst_path, img)

print("✅ Dataset creation complete.")


✅ Dataset creation complete.


In [12]:
from sklearn.model_selection import train_test_split
import numpy as np
import cv2
import os

IMG_SIZE = (64, 64)
DATA_PATH = r"C:\Users\USER\Documents\ML and Apps Project\eye_dataset"

X, y = [], []

label_map = {"open": 0, "closed": 1}

for label_name in ["open", "closed"]:
    folder = os.path.join(DATA_PATH, label_name)
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, IMG_SIZE)
            X.append(img)
            y.append(label_map[label_name])

X = np.array(X).reshape(-1, 64, 64, 1) / 255.0
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 1)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)


Epoch 1/10


C:\Users\USER\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 70ms/step - accuracy: 0.7363 - loss: 0.5591 - val_accuracy: 0.9067 - val_loss: 0.2504
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.9134 - loss: 0.2369 - val_accuracy: 0.9229 - val_loss: 0.1835
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.9449 - loss: 0.1721 - val_accuracy: 0.8661 - val_loss: 0.3070
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - accuracy: 0.9376 - loss: 0.1899 - val_accuracy: 0.9412 - val_loss: 0.1444
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 65ms/step - accuracy: 0.9528 - loss: 0.1233 - val_accuracy: 0.9473 - val_loss: 0.1225
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 64ms/step - accuracy: 0.9729 - loss: 0.0863 - val_accuracy: 0.9635 - val_loss: 0.0958
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.9737 - loss: 0.0743 - val_accuracy: 0.9635 - val_loss: 0.0900
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 4s 61ms/step - accuracy: 0.9692 - loss: 0.0836 - val_accuracy: 0.9635 - val_loss: 0.

In [16]:
model.save("eye_state_model_realistic.h5")
loss, acc = model.evaluate(X_test, y_test)
print(f"✅ Test Accuracy: {acc*100:.2f}%")


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9702 - loss: 0.0737
✅ Test Accuracy: 96.96%


In [18]:
model.save("eye_state_model_realistic.h5")
from tensorflow.keras.models import load_model

In [20]:
IMG_SIZE = (64, 64)
model = load_model("eye_state_model_realistic.h5")

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True)

eye_indices = {'left': [33, 133], 'right': [362, 263]}

def predict_eye_state(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Could not read image.")
        return

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if not results.multi_face_landmarks:
        print("No face detected.")
        return

    h, w, _ = img.shape
    predictions = []

    for face_landmarks in results.multi_face_landmarks:
        for side in ['left', 'right']:
            pt1 = face_landmarks.landmark[eye_indices[side][0]]
            pt2 = face_landmarks.landmark[eye_indices[side][1]]

            x1, y1 = int(pt1.x * w), int(pt1.y * h)
            x2, y2 = int(pt2.x * w), int(pt2.y * h)

            margin = 10
            x1, y1 = max(x1 - margin, 0), max(y1 - margin, 0)
            x2, y2 = min(x2 + margin, w), min(y2 + margin, h)

            eye_img = img[y1:y2, x1:x2]
            if eye_img.size == 0:
                continue

            eye_gray = cv2.cvtColor(eye_img, cv2.COLOR_BGR2GRAY)
            eye_resized = cv2.resize(eye_gray, IMG_SIZE)
            eye_input = eye_resized.reshape(1, 64, 64, 1) / 255.0

            prob = model.predict(eye_input)[0][0]
            label = 'Closed' if prob > 0.5 else 'Open'
            predictions.append(label)

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    if len(predictions) == 0:
        print("No eyes detected.")
    elif predictions.count('Closed') >= 2:
        print("⚠️ Drowsiness Detected!")
    else:
        print("✅ Eyes appear open.")

    cv2.imshow("Eye State Prediction", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return predictions

In [22]:
predict_eye_state(r"C:\Users\USER\Documents\ML and Apps Project\Drowsiness_dataset_new\Drowsiness_dataset_new\test\yawn\526.jpg")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
✅ Eyes appear open.


['Open', 'Open']